# Outcome Dynamics — individual trajectories

Single simulation without treatment. Two plots: one per outcome, all firm trajectories + mean.

In [ ]:
import sys
import copy
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

from data_generation.simulator import DataSimulator
from data_generation.data_config import DATA_CONFIG

import pandas as pd

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
sim   = DataSimulator(DATA_CONFIG)
panel = sim.simulate()
print(f"{panel['id_firma'].nunique()} firms, {panel['t'].nunique()} periods")

panel.head()

In [ ]:
# Separamos en tratados, controles y nini para tenerlos a mano
treated_mask = panel['tratado_en_t'] == True
treated = panel[treated_mask]

control_mask = panel['control_en_t'] == True
control = panel[control_mask]

nini_mask = ~panel['id_firma'].isin(
    treated['id_firma'].unique().tolist() + control['id_firma'].unique().tolist()
)
nini = panel[nini_mask]

In [ ]:
outcomes = list(DATA_CONFIG['dinamica_outcomes'].keys())

GROUP_COLOR = {'tratado': 'crimson', 'control': 'steelblue', 'nini': 'silver'}
GROUP_ALPHA = {'tratado': 0.25,      'control': 0.25,        'nini': 0.12}

treated_id_set = set(treated['id_firma'].unique().tolist())
control_id_set = set(control['id_firma'].unique().tolist())
nini_id_set    = set(nini['id_firma'].unique().tolist())

def firm_group(firm_id):
    if firm_id in treated_id_set: return 'tratado'
    if firm_id in control_id_set: return 'control'
    return 'nini'

group_masks = {
    'tratado': panel['id_firma'].isin(treated_id_set),
    'control': panel['id_firma'].isin(control_id_set),
    'nini':    nini_mask,
}

for outcome in outcomes:
    fig, ax = plt.subplots(figsize=(10, 5))

    for firm_id, firm_df in panel.groupby('id_firma'):
        g = firm_group(firm_id)
        ax.plot(firm_df['t'], firm_df[outcome],
                color=GROUP_COLOR[g], alpha=GROUP_ALPHA[g], linewidth=0.8)

    ax.set_title(outcome, fontsize=12)
    ax.set_xlabel('Período (t)', fontsize=11)
    ax.set_ylabel(outcome, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
outcomes = list(DATA_CONFIG['dinamica_outcomes'].keys())

GROUP_COLOR = {'tratado': 'crimson', 'control': 'steelblue', 'nini': 'silver'}
GROUP_ALPHA = {'tratado': 0.25,      'control': 0.25,        'nini': 0.12}

treated_id_set = set(treated['id_firma'].unique().tolist())
control_id_set = set(control['id_firma'].unique().tolist())
nini_id_set    = set(nini['id_firma'].unique().tolist())

def firm_group(firm_id):
    if firm_id in treated_id_set: return 'tratado'
    if firm_id in control_id_set: return 'control'
    return 'nini'

group_masks = {
    'tratado': panel['id_firma'].isin(treated_id_set),
    'control': panel['id_firma'].isin(control_id_set),
    'nini':    nini_mask,
}

for outcome in outcomes:
    fig, ax = plt.subplots(figsize=(10, 5))

    for g, mask in group_masks.items():
        mean_traj = panel[mask].groupby('t')[outcome].mean()
        ax.plot(mean_traj.index, mean_traj.values,
                color=GROUP_COLOR[g], linewidth=2.2, label=g)

    ax.set_title(outcome, fontsize=12)
    ax.set_xlabel('Período (t)', fontsize=11)
    ax.set_ylabel(outcome, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()